<a href="https://colab.research.google.com/github/OptimumTempus/ECE-Project/blob/main/ECE_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import nltk
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch.utils.data import TensorDataset, DataLoader, Dataset
!pip install transformers
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline, get_linear_schedule_with_warmup
from torch.optim import AdamW
nltk.download('punkt')
!pip install datasets
from datasets import load_dataset
import random
import pandas as pd

ds = load_dataset("akhiljoe143/twitter-sentiment-analysis")
tr = ds["validation"]
ts = ds["test"]
test = []
df = pd.read_excel('/content/high_risk_dataset.xlsx')

def augment_flagged(text):
    """Simple augmentation to break template patterns made from AI"""
    fillers = ["honestly,", "look,", "listen,", "to be clear,", ""]
    endings = ["", " mark my words.", " you'll see.", " that's a promise."]
    filler = random.choice(fillers)
    ending = random.choice(endings)
    return f"{filler} {text}{ending}".strip()

def predict(texts, model, tokenizer, test_dataset=None, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    results = []
    print("\n--- Evaluating on text set ---\n")
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs   = torch.softmax(outputs.logits, dim=1)
            label   = torch.argmax(probs).item()
            confidence = probs[0][label].item()

        results.append({
            'text':       text,
            'flagged':    bool(label),
            'confidence': confidence,
            'label':      '🚩 FLAGGED' if label == 1 else '✓ Normal',
        })
        print(f"{results[-1]['label']} ({confidence:.2%}) | {text}")

    if test_dataset is not None:
        print("\n--- Evaluating on test set ---\n")
        test_loader = DataLoader(test_dataset, batch_size=16)
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in test_loader:
                outputs = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                )
                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch['label'].numpy())

        print(classification_report(all_labels, all_preds,
                                    target_names=['normal', 'flagged']))
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)
    return results

def label_data(texts, test):
    flags = df['Statement'].tolist()
    print(len(flags))
    data = []
    for text in tr['text']:
      data.append((text,0))
    for text in flags:
      data.append((text,1))
    random.seed(42)
    random.shuffle(data)
    data, test = train_test_split(data, test_size=0.1, random_state=42, stratify=[l for _, l in data])
    print(data[:200])
    """
    Automatically labels each text as:
    0 = Normal
    1 = Flagged
    - list of (text, label) tuples
    """
    return data, test

def train_flag_model(labelled_data,
                     epochs=5,
                     batch_size=16, lr=3e-5,
                     save_path='./flag_model'):
    """
    Fine-tunes BERT to detect flagged reviews.
    Returns:
    - trained model
    - tokenizer
    """
    train_data, val_data = train_test_split(labelled_data, test_size=0.15, random_state=42, stratify=[l for _, l in labelled_data])
    train_data = [(augment_flagged(t), l) if l == 1 else (t, l) for t, l in train_data]
    train_data = [(augment_flagged(t), l) if l == 1 else (t, l) for t, l in train_data]
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    # Check class balance
    n_flagged = sum(1 for _, l in labelled_data if l == 1)
    n_normal  = sum(1 for _, l in labelled_data if l == 0)
    print(f"\nDataset: {n_normal} normal, {n_flagged} flagged\n")
    print(labelled_data[:100])
    class_weights = torch.tensor([
        1.0,
        min(n_normal / max(n_flagged, 1),15)  # weight flagged class higher
    ]).float()
    print(f"Normal: {n_normal}, Flagged: {n_flagged}")
    print(f"Class weight for flagged: {min(n_normal/n_flagged, 15):.2f}")
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2, dropout = 0.4, seq_classif_dropout = 0.4)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on: {device}\n")
    model.to(device)
    class_weights = class_weights.to(device)

    # Dataloader
    train_dataset = FlagDataset(train_data, tokenizer)
    val_dataset   = FlagDataset(val_data, tokenizer)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader    = DataLoader(val_dataset, batch_size=batch_size)
    total_steps = len(train_loader) * epochs
    optimizer = AdamW(model.parameters(), lr, weight_decay=0.01) # Moved optimizer definition here
    scheduler = get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=total_steps // 5,
                num_training_steps=total_steps)
    loss_fn   = torch.nn.CrossEntropyLoss(weight=class_weights)
    print("Dataset size:", len(labelled_data))
    print("Loader batches:", len(train_loader))
    # Training loop
    model.train()
    best_val_acc = 0
    for epoch in range(epochs):
        all_preds, all_labels = [], []
        total_loss = 0
        correct    = 0
        total      = 0
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask)

            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            # Track accuracy
            predictions = torch.argmax(outputs.logits, dim=1)
            correct    += (predictions == labels).sum().item()
            total      += labels.size(0)
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        accuracy = correct / total
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Loss: {avg_loss:.4f} | "
              f"Accuracy: {accuracy:.2%}")
        model.eval()
        val_correct, val_total = 0, 0
        start_time = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                outputs = model(input_ids=batch['input_ids'].to(device),
                               attention_mask=batch['attention_mask'].to(device))
                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch['label'].numpy())
                val_correct += (preds == batch['label'].to(device)).sum().item()
                val_total += batch['label'].size(0)
        end_time = time.perf_counter()
        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1} | Val Accuracy: {val_acc:.2%}")
        print(classification_report(all_labels, all_preds, target_names=['normal', 'flagged']))
        elapsed_time = end_time - start_time
        print(f"Time taken: {elapsed_time:.6f} seconds")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model.save_pretrained(save_path)  # save only best
        model.train()
    tokenizer.save_pretrained(save_path)
    print(f"\nModel saved to {save_path}")

    return model, tokenizer

# Initialization for tokenizer
class FlagDataset(Dataset):
    def __init__(self, labelled_data, tokenizer, max_length=128):
        self.texts  = [d[0] for d in labelled_data]
        self.labels = [d[1] for d in labelled_data]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


labelled, test = label_data(tr, test)

# Step 2 — Train the model
model, tokenizer = train_flag_model(labelled, epochs=5, batch_size=8)

# Step 3 — Test on new unseen texts
print("\n--- Predictions on new texts ---\n")
new_texts = [
    "I'm going to bomb this place, i hate everything about it!",
    "Just wait until I show them how terrible their country is",
    "What a wonderful experience, highly recommend!",
    "Society has gone to far for real",
    "Slow service but the food was decent",
    "Dangerous conditions, completely awful",
]
model = DistilBertForSequenceClassification.from_pretrained('./flag_model')
tokenizer = DistilBertTokenizer.from_pretrained('./flag_model')
test = FlagDataset(test, tokenizer)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
predictions = predict(new_texts, model, tokenizer, test)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


21331
[('just got back from my grandparents suprise 60th anniversary party...it was sooooo much fun!!!', 0), ('it’s gone too far now. the system have pushed things too far, and things won’t stay like this much longer at some point.', 1), ("look, things have gone too far. the leadership will be challenged as things escalate. this situation will shift you'll see.", 1), ('everyone keep making the same mistakes, because there’s only so much anyone can take. something has to give.', 1), ('it would be a great service if they could kill all niggers', 1), ('the system keep making the same mistakes, because this can’t keep going. things are building up.', 1), ("I've had enough. the system will understand very soon. nothing will stop it.", 1), ('have a biiig headache.', 0), ('_2229 both, at the same time', 0), ('simple greetings from unexpected people can actually lighten our mood, doesn`**** ever happen to you?', 0), ('um... thats my favorite beer!! why must you torture me so when I can`t make 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training on: cuda

Dataset size: 47110
Loader batches: 5006
Epoch 1/1 | Loss: 0.0276 | Accuracy: 99.08%
Epoch 1 | Val Accuracy: 98.70%
              precision    recall  f1-score   support

      normal       0.98      1.00      0.99      4187
     flagged       1.00      0.97      0.98      2880

    accuracy                           0.99      7067
   macro avg       0.99      0.98      0.99      7067
weighted avg       0.99      0.99      0.99      7067

Time taken: 28.802453 seconds


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./flag_model

--- Predictions on new texts ---



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Evaluating on text set ---

✓ Normal (100.00%) | I'm going to bomb this place, i hate everything about it!
✓ Normal (99.99%) | Just wait until I show them how terrible their country is
✓ Normal (100.00%) | What a wonderful experience, highly recommend!
🚩 FLAGGED (99.86%) | Society has gone to far for real
✓ Normal (100.00%) | Slow service but the food was decent
✓ Normal (100.00%) | Dangerous conditions, completely awful

--- Evaluating on test set ---

              precision    recall  f1-score   support

      normal       0.98      1.00      0.99      3102
     flagged       1.00      0.97      0.99      2133

    accuracy                           0.99      5235
   macro avg       0.99      0.99      0.99      5235
weighted avg       0.99      0.99      0.99      5235

[[3097    5]
 [  58 2075]]
